<a href="https://colab.research.google.com/github/aavarela/SPBD_Labs/blob/main/projeto2/report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPBD 2526 Project 2

version 0.1 (27 Nov 2025)

# Context
The project scenario involves a dataset of taxi rides, collected circa 2013, in the New York city area.

This project scenario is inspired by the ACM DEBS 2015 Grand Challenge.

### Taxi Rides

Each completed taxi ride comprises a number of attributes, separated by commas, as follows:

| Attribute   | Description |
| :---        |        :--- |
|medallion| an md5sum of the identifier of the taxi - vehicle bound|
|hack_license| an md5sum of the identifier for the taxi license|
|pickup_datetime| time when the passenger(s) were picked up|
|dropoff_datetime| time when the passenger(s) were dropped off|
|trip_time_in_secs| duration of the trip|
|trip_distance| trip distance in miles|
|pickup_longitude| longitude coordinate of the pickup location|
|pickup_latitude| latitude coordinate of the pickup location|
|dropoff_longitude| longitude coordinate of the drop-off location|
|dropoff_latitude| latitude coordinate of the drop-off location|
|payment_type| the payment method - credit card or cash|
|fare_amount| fare amount in dollars|
|surcharge| surcharge in dollars|
|mta_tax| tax in dollars|
|tip_amount| tip in dollars|
|tolls_amount| bridge and tunnel tolls in dollars|
|total_amount| total paid amount in dollars|

## Real-time Stream

For this assignment the data is presented as a "real-time" stream, made available through Apache Kafka, with the following considerations:

The stream is the same data as the original, but the timestamps (pickup_datetime and dropoff_datime) are adjusted to reflect the current time, not the date of acquisition (2013).

The stream can be replayed faster than realtime, by supplying a **speedup factor**. For example, a speedup of 60, means that 1 second in realtime, corresponds to 60 virtual seconds in the dataset.

The trip duration is not affected by the speedup factor; the same applies to the difference between dropoff_datime and pickup_datetime.


An example of the JSON encoding of each taxi ride event is the following:
```json
{"medallion": "045C16AC567D6B720C793C156F480CC4", "hack_license": "D39A22C155B4C912D0D09039BF3892B1", "pickup_datetime": "2025-11-27 16:01:19.871522", "dropoff_datetime": "2025-11-27 16:16:19.871522", "trip_time_in_secs": 900, "trip_distance": 3.59, "pickup_longitude": -74.013298, "pickup_latitude": 40.703938, "dropoff_longitude": -74.00193, "dropoff_latitude": 40.739403, "payment_type": "CRD", "fare_amount": 14.0, "surcharge": 0.5, "mta_tax": 0.5, "tip_amount": 2.9, "tolls_amount": 0.0, "total_amount": 17.9}
```

##Environment configuration
The collab environment loads with pyspark 4.0.1 by default. This version of pyspark is not compatible with Kafka 3.7.2, which leads to errors while loading the Kafka stream. Replacing pyspark 4.0.1 with pyspark 3.5.1 solves this issue. To work around an error with dataproc-spark-connect 1.0.1, which depends on pysaprk 4.0.1, we first uninstall dataproc-spark-connect 1.0.1.

In [2]:
#@title Uninstall dataproc-spark-connect 1.0.1 and current pyspark (if any) and install 3.5.1
!pip uninstall -y dataproc-spark-connect
!pip uninstall -y pyspark
!pip install pyspark==3.5.1

Found existing installation: pyspark 4.0.1
Uninstalling pyspark-4.0.1:
  Successfully uninstalled pyspark-4.0.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 12.7 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=3dda5dd8a42e998edd21952423b1fa781d79b989204048bf57f9e1bff829ee43
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9


In [3]:
#@title Download dataset

!wget -q -O taxi_rides_1pc.csv.gz https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&st=iurdwnwj&dl=0

In [4]:
#@title Install & Launch Kafka
%%bash
KAFKA_VERSION=3.7.2
KAFKA=kafka_2.12-$KAFKA_VERSION
wget -q -O /tmp/$KAFKA.tgz https://dlcdn.apache.org/kafka/$KAFKA_VERSION/$KAFKA.tgz
tar xfz /tmp/$KAFKA.tgz
wget -q -O $KAFKA/config/server1.properties - https://github.com/smduarte/spbd-2526/raw/refs/heads/main/docs/labs/projs/server1.properties

UUID=`$KAFKA/bin/kafka-storage.sh random-uuid`
$KAFKA/bin/kafka-storage.sh format -t $UUID -c $KAFKA/config/server1.properties
$KAFKA/bin/kafka-server-start.sh -daemon $KAFKA/config/server1.properties

Formatting /tmp/kraft-combined-logs with metadata.version 3.7-IV4.


### Kafka publisher
This a small python Kafka producer that publishes the rides to Kafka. It simulates the source of events, represented by the taxis as they report the rides that have concluded.

* The Kafka server is accessible @localhost:9092
* The events are published to the `taxis_json` topic
* By default, events are published 60x faster than realtime relative to the orignal timestamps.


In [5]:
#@title Start Kafka Publisher
!pip --quiet install kafka-python dataclasses
!wget -q -O kafka-publisher.py https://raw.githubusercontent.com/smduarte/spbd-2526/refs/heads/main/docs/labs/projs/kafka-publisher.py

!nohup python kafka-publisher.py --topic taxis_json --speedup 60 --filename taxi_rides_1pc.csv.gz 2> /dev/null &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 6.1 MB/s eta 0:00:00


The python code below shows the basics needed to process JSON data from Kafka source using PySpark.

Spark Streaming python documentation is found [here](https://spark.apache.org/docs/latest/api/python/reference/pyspark.streaming.html)

---
#### PySpark Kafka Stream Example


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

def dumpBatchDF(df, epoch_id):
    df.show(20, False)

spark = SparkSession \
    .builder \
    .appName('Kafka Spark Structured Streaming Example') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1') \
    .getOrCreate()

lines = spark \
  .readStream \
  .format('kafka') \
  .option('kafka.bootstrap.servers', 'localhost:9092') \
  .option('subscribe', 'taxis_json') \
  .option('startingOffsets', 'earliest') \
  .load() \
  .selectExpr('CAST(value AS STRING)')

query = lines \
    .writeStream \
    .outputMode('append') \
    .foreachBatch(dumpBatchDF) \
    .start()

query.awaitTermination(600)
query.stop()
spark.stop()

Streaming output truncated to the last 5000 lines.
|value                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/usr/local/lib/python3.12/dist-packages/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipython-input-371947984.py", line 6, in dumpBatchDF
    df.show(20, False)
  File "/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py", line 945, in show
    print(self._show_string(n, truncate, vertical))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py", line 976, in _show_string
    ret

# Objectives

The main objective of this second assignment is to revisit this dataset considering that, in addition, to the original archived data, we have a realtime stream of what is happening **now**.

Some broad approaches to pursue in this second assignment can be:

1. How could the analysis selected and presented in the first assignment be refined and/or enriched by including realtime knowlege.

2. Can you think of an application/scenario that might be benefit or made possible by knowing what is happening now, compared to is normal in the archived dataset.

3. How does what is happening now, compare to what is present in the archived dataset, possibly, when evaluated in some abstract way in terms of statistical
indicators (without an application in mind).

This second assignment can be seen as complementary to the first assignment. As such, it is meant as a shorter and more focused effort. It is not necessary to repeat explanations. Delivery will comprise a single notebook. The bulk of the contents should be on discussing the direction/approach being pursued and presenting any results and their associated code. Details regarding AI usage should be provided as an Addendum at the end of the notebook.

# Requeriments

Code will need to leverage Spark Structured Streaming, in some way.

---
### Execution

Groups of up to 2 or 3 elements: 1 or 2 Humans, 1 AI agent.

### Delivery Format

A google forms will be provided closer to the deadline for delivery purposes.

---

# Grading

Grading will take into consideration the overall presentation quality of the report and its technical merit.

Use of AI tools is allowed and even recommended, for example, to enrich the presentation with visual elements. However, all prompts used need to be reported.

Showing effective use of AI agents will be graded positively.

---

### Deadline

December 20, 2025. Penalty of 0.01/20 for each day late. Accumulates til grade reaches 9.5/20. Deliveries past December 31, 2025 will not be considered.